### 0) Import

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures

### 1) Answer the questions

#### a) analytical solution to the regression problem
$$
(y - X\theta)^2 = (y - X\theta)^T(y - X\theta) = y^Ty - y^TX\theta - \theta^TX^Ty + \theta^TX^TX\theta
$$
$$
0 = \nabla_{\theta}(y^Ty - y^TX\theta - \theta^TX^Ty + \theta^TX^TX\theta) = 0 - y^TX - X^Ty + 2X^TX\theta 
$$
$$
-2X^Ty + 2X^TX\theta = 0
$$
$$
\text{Ответ}: \theta = (X^TX)^{-1}X^Ty
$$

#### b) What changes in the solution when L1 and L2 regularizations are added to the loss function.
Дополнительный член в функции потерь, например, для
$$ 
L_2: \theta = (X^TX+\lambda I)^{-1}X^Ty
$$
$$
\min_{\boldsymbol{\theta}} \sum_{i=1}^N L \left( f(x_i, \boldsymbol{\theta}), y_i \right) + \lambda R(\boldsymbol{\theta})
$$

#### c) Explain why L1 regularization is often used to select features. Why are there many weights equal to 0 after the model is fit?
Поскольку каждый ненулевой коэффициент увеличивает штрафную функцию, слабые признаки должны иметь нулевые коэффициенты.

#### d) Explain how you can use the same models (Linear regression, Ridge, etc.) but make it possible to fit nonlinear dependencies.
Необходимо использовать нелинейные подходы к работе функции, например, log(), sin() и т.д.

### 2) Introduction

In [2]:
data_train = pd.read_json('../datasets/train.json')
data_test = pd.read_json('../datasets/test.json')

In [3]:
data_train.reset_index(inplace=True)
data_test.reset_index(inplace=True)
df_train = pd.DataFrame()
df_test = pd.DataFrame()

In [4]:
data_train = data_train[(data_train['bathrooms'] >= data_train['bathrooms'].quantile(0.01)) & (data_train['bathrooms'] <= data_train['bathrooms'].quantile(0.99))]
data_train = data_train[(data_train['bedrooms'] >= data_train['bedrooms'].quantile(0.01)) & (data_train['bedrooms'] <= data_train['bedrooms'].quantile(0.99))]
data_train = data_train[(data_train['price'] >= data_train['price'].quantile(0.01)) & (data_train['price'] <= data_train['price'].quantile(0.99))]

In [5]:
data_test = data_test[(data_test['bathrooms'] >= data_train['bathrooms'].quantile(0.01)) & (data_test['bathrooms'] <= data_train['bathrooms'].quantile(0.99))]
data_test = data_test[(data_test['bedrooms'] >= data_train['bedrooms'].quantile(0.01)) & (data_test['bedrooms'] <= data_train['bedrooms'].quantile(0.99))]
data_test = data_test[(data_test['price'] >= data_train['price'].quantile(0.01)) & (data_test['price'] <= data_train['price'].quantile(0.99))]

### 3) Intro data analysis

In [6]:
all_features = []
for index, row in data_train.iterrows():
    for feature in row['features']:
        features = feature.replace('[', '').replace(']', '').replace("'", '').replace('"', '').replace(' ', '')
        all_features.append(features)

In [7]:
print(len(set(all_features)))

1516


In [8]:
all_features = Counter(all_features).most_common(20)
feature_list = [feature for feature, _ in all_features]
all_features

[('Elevator', 24956),
 ('HardwoodFloors', 22848),
 ('CatsAllowed', 22844),
 ('DogsAllowed', 21383),
 ('Doorman', 20162),
 ('Dishwasher', 19787),
 ('NoFee', 17435),
 ('LaundryinBuilding', 15855),
 ('FitnessCenter', 12770),
 ('Pre-War', 8878),
 ('LaundryinUnit', 8195),
 ('RoofDeck', 6328),
 ('OutdoorSpace', 5015),
 ('DiningRoom', 4734),
 ('HighSpeedInternet', 4180),
 ('Balcony', 2819),
 ('SwimmingPool', 2603),
 ('LaundryInBuilding', 2554),
 ('NewConstruction', 2469),
 ('Terrace', 2109)]

In [9]:
for feature in feature_list:
    df_train[feature] = data_train['features'].apply(lambda x: 1 if feature in x else 0)
    df_test[feature] = data_test['features'].apply(lambda x: 1 if feature in x else 0)
df_train['bathrooms'] = data_train['bathrooms']
df_train['bedrooms'] = data_train['bedrooms']
df_test['bathrooms'] = data_test['bathrooms']
df_test['bedrooms'] = data_test['bedrooms']
df_train

,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,...,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace,bathrooms,bedrooms
0,0,0,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,1.0,1
1,1,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,2
2,1,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,2
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1.5,3
4,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49347,1,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,3
49348,1,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1.0,2
49349,1,0,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,1.0,1
49350,0,0,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,1.0,2


### 4) Models implementation — Linear regression

In [10]:
X_train = df_train
y_train = data_train['price']
X_test = df_test
y_test = data_test['price']

#### Model with an analytical solution

In [11]:
# Little example
# y = 1 + x1 * 2 + x2 * 3 + x3 * 4
# X = np.array([[1, 1, 1],
#               [1, 2, 3],
#               [2, 3, 4]])

# y = np.array([10, 21, 30])

In [12]:
class LinearRegressionAnalyttical:
    def __init__(self):
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        
        bias = np.ones((X.shape[0], 1))
        X = np.concatenate((bias, X), axis=1)
        weights = np.linalg.pinv(X.T @ X) @ X.T @ y

        self.bias = weights[0]
        self.weights = weights[1:]

    def predict(self, X):
        X = np.array(X)
        return (X @ self.weights + self.bias).reshape(-1)

In [13]:
# analytical_model = LinearRegressionAnalyttical()
# analytical_model.fit(X, y)
# y_pred = analytical_model.predict(X)
# print(y_pred)

#### A model with non-stochastic gradient descent

$$
MSE: \rho = \frac{1}{n+1} \left( \sum_{i=0}^n(y_i-\hat{y_i})^2 \right)
$$
$$
\hat{y} = \sum_{j=0}^mw_jx_j = w_0 + w_1x_1 + w_2x_2 + ... + w_mx_m
$$
$$
\frac{\partial \rho}{\partial w_k} = \frac{2}{n+1} \left( \sum_{i=0}^n(\hat{y_i}-y_i) \right)x_{i,k}, k=0,...,m
$$

In [14]:
class LinearRegressionGD:
    def __init__(self, learning_rate=0.01, epochs=1000):
        self.weights = None
        self.bias = None
        self.lr = learning_rate
        self.epochs = epochs

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)
        n_samples, m_features = X.shape

        bias = np.ones((n_samples, 1))
        X = np.concatenate((bias, X), axis=1)
        self.weights = np.zeros((m_features+1, 1))
        
        for _ in range(self.epochs):
            y_pred = X @ self.weights
            gradient = (2 / n_samples) * (X.T @ (y_pred - y))
            self.weights -= self.lr * gradient

        self.bias = self.weights[0]
        self.weights = self.weights[1:]

    def predict(self, X):
        X = np.array(X)
        return (X @ self.weights + self.bias).reshape(-1)

In [15]:
# gd_model = LinearRegressionGD()
# gd_model.fit(X, y)
# y_pred = gd_model.predict(X)
# print(y_pred)

#### A model with stochastic gradient descent

In [16]:
class LinearRegressionDSGD:
    def __init__(self, learning_rate=0.01, epochs=1000, batch_size=128, seed=21):
        self.weights = None
        self.bias = None
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.seed = seed

    def fit(self, X, y):
        np.random.seed(self.seed)
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)

        n_samples, m_features = X.shape
        bias = np.ones((n_samples, 1))
        X = np.concatenate((bias, X), axis=1)

        self.weights = np.zeros((m_features+1, 1))
        for _ in range(self.epochs):
            indices = np.random.permutation(n_samples)
            X_shuffle = X[indices]
            y_shuffle = y[indices]

            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffle[i:i+self.batch_size]
                y_batch = y_shuffle[i:i+self.batch_size]
                y_pred = X_batch @ self.weights
                gradient = (2 / self.batch_size) * (X_batch.T @ (y_pred - y_batch))
    
                self.weights -= self.lr * gradient

        self.bias = self.weights[0]
        self.weights = self.weights[1:]

    def predict(self, X):
        X = np.array(X)
        return (X @ self.weights + self.bias).reshape(-1)

In [17]:
def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tol = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tol

In [18]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred_train_def = model.predict(X_train)
y_pred_test_def = model.predict(X_test)

In [19]:
dsgd_model = LinearRegressionDSGD()
dsgd_model.fit(X_train, y_train)
y_pred_train_dsgd = dsgd_model.predict(X_train)
y_pred_test_dsgd = dsgd_model.predict(X_test)

In [20]:
result_MAE = pd.DataFrame(columns=['model', 'train', 'test'])
result_RMSE = pd.DataFrame(columns=['model', 'train', 'test'])
result_R2 = pd.DataFrame(columns=['model', 'train', 'test'])

In [21]:
result_MAE.loc[0] = ['Linreg default', mae(y_train, y_pred_train_def), mae(y_test, y_pred_test_def)]
result_RMSE.loc[0] = ['Linreg default', rmse(y_train, y_pred_train_def), rmse(y_test, y_pred_test_def)]
result_R2.loc[0] = ['Linreg default', r2_score(y_train, y_pred_train_def), r2_score(y_test, y_pred_test_def)]

In [22]:
result_MAE.loc[1] = ['Linreg DSGD', mae(y_train, y_pred_train_dsgd), mae(y_test, y_pred_test_dsgd)]
result_RMSE.loc[1] = ['Linreg DSGD', rmse(y_train, y_pred_train_dsgd), rmse(y_test, y_pred_test_dsgd)]
result_R2.loc[1] = ['Linreg DSGD', r2_score(y_train, y_pred_train_dsgd), r2_score(y_test, y_pred_test_dsgd)]

In [23]:
result_MAE

,model,train,test
0,Linreg default,694.523900,662.951989
1,Linreg DSGD,693.439133,661.769381


In [24]:
result_RMSE

,model,train,test
0,Linreg default,968.869323,890.118262
1,Linreg DSGD,968.902286,889.826433


In [25]:
result_R2

,model,train,test
0,Linreg default,0.549336,0.512232
1,Linreg DSGD,0.549305,0.512552


### 5) Regularized models implementation — Ridge, Lasso, ElasticNet

In [26]:
class RidgeRegression(LinearRegressionDSGD):
    def __init__(self, alpha=0.01, **kwargs):
        super().__init__(**kwargs)
        self.alpha = alpha

    def fit(self, X, y):
        np.random.seed(self.seed)
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)

        n_samples, m_features = X.shape
        bias = np.ones((n_samples, 1))
        X = np.concatenate((bias, X), axis=1)

        self.weights = np.zeros((m_features+1, 1))
        for _ in range(self.epochs):
            indices = np.random.choice(n_samples, size=self.batch_size, replace=False)
            X_batch = X[indices]
            y_batch = y[indices]

            y_pred = X_batch @ self.weights
            gradient = (2 / self.batch_size) * (X_batch.T @ (y_pred - y_batch))
            l2 = 2 * self.alpha * self.weights
            self.weights -= self.lr * (gradient + l2)

        self.bias = self.weights[0]
        self.weights = self.weights[1:]

In [27]:
model_ridge = Ridge()
model_ridge.fit(X_train, y_train)
y_pred_train_ridge = model_ridge.predict(X_train)
y_pred_test_ridge = model_ridge.predict(X_test)

In [28]:
model_ridreg = RidgeRegression()
model_ridreg.fit(X_train, y_train)
y_pred_train_ridreg = model_ridreg.predict(X_train)
y_pred_test_ridreg = model_ridreg.predict(X_test)

In [29]:
result_MAE.loc[2] = ['Ridge default', mae(y_train, y_pred_train_ridge), mae(y_test, y_pred_test_ridge)]
result_RMSE.loc[2] = ['Ridge default', rmse(y_train, y_pred_train_ridge), rmse(y_test, y_pred_test_ridge)]
result_R2.loc[2] = ['Ridge default', r2_score(y_train, y_pred_train_ridge), r2_score(y_test, y_pred_test_ridge)]

In [30]:
result_MAE.loc[3] = ['Ridge', mae(y_train, y_pred_train_ridreg), mae(y_test, y_pred_test_ridreg)]
result_RMSE.loc[3] = ['Ridge', rmse(y_train, y_pred_train_ridreg), rmse(y_test, y_pred_test_ridreg)]
result_R2.loc[3] = ['Ridge', r2_score(y_train, y_pred_train_ridreg), r2_score(y_test, y_pred_test_ridreg)]

In [31]:
class LassoRegression(LinearRegressionDSGD):
    def __init__(self, alpha=0.001, **kwargs):
        super().__init__(**kwargs)
        self.alpha = alpha

    def fit(self, X, y):
        np.random.seed(self.seed)
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)

        n_samples, m_features = X.shape
        bias = np.ones((n_samples, 1))
        X = np.concatenate((bias, X), axis=1)

        self.weights = np.zeros((m_features+1, 1))
        for _ in range(self.epochs):
            indices = np.random.choice(n_samples, size=self.batch_size, replace=False)
            X_batch = X[indices]
            y_batch = y[indices]

            y_pred = X_batch @ self.weights
            gradient = (2 / self.batch_size) * (X_batch.T @ (y_pred - y_batch))
            l1 = self.alpha * np.sign(self.weights)
            self.weights -= self.lr * (gradient + l1)

        self.bias = self.weights[0]
        self.weights = self.weights[1:]

In [32]:
model_lasso = Lasso()
model_lasso.fit(X_train, y_train)
y_pred_train_lasso = model_lasso.predict(X_train)
y_pred_test_lasso = model_lasso.predict(X_test)

In [33]:
model_lasreg = LassoRegression()
model_lasreg.fit(X_train, y_train)
y_pred_train_lasreg = model_lasreg.predict(X_train)
y_pred_test_lasreg = model_lasreg.predict(X_test)

In [34]:
result_MAE.loc[4] = ['Lasso default', mae(y_train, y_pred_train_lasso), mae(y_test, y_pred_test_lasso)]
result_RMSE.loc[4] = ['Lasso default', rmse(y_train, y_pred_train_lasso), rmse(y_test, y_pred_test_lasso)]
result_R2.loc[4] = ['Lasso default', r2_score(y_train, y_pred_train_lasso), r2_score(y_test, y_pred_test_lasso)]

In [35]:
result_MAE.loc[5] = ['Lasso', mae(y_train, y_pred_train_lasreg), mae(y_test, y_pred_test_lasreg)]
result_RMSE.loc[5] = ['Lasso', rmse(y_train, y_pred_train_lasreg), rmse(y_test, y_pred_test_lasreg)]
result_R2.loc[5] = ['Lasso', r2_score(y_train, y_pred_train_lasreg), r2_score(y_test, y_pred_test_lasreg)]

In [36]:
class ElasticNetRegression(LinearRegressionDSGD):
    def __init__(self, alpha1=0.001, alpha2=0.001, **kwargs):
        super().__init__(**kwargs)
        self.alpha1 = alpha1
        self.alpha2 = alpha2

    def fit(self, X, y):
        np.random.seed(self.seed)
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)

        n_samples, m_features = X.shape
        bias = np.ones((n_samples, 1))
        X = np.concatenate((bias, X), axis=1)

        self.weights = np.zeros((m_features+1, 1))
        for _ in range(self.epochs):
            indices = np.random.choice(n_samples, size=self.batch_size, replace=False)
            X_batch = X[indices]
            y_batch = y[indices]

            y_pred = X_batch @ self.weights
            gradient = (2 / self.batch_size) * (X_batch.T @ (y_pred - y_batch))
            l1 = self.alpha1 * np.sign(self.weights)
            l2 = 2 * self.alpha2 * self.weights
            self.weights -= self.lr * (gradient + l1 + l2)

        self.bias = self.weights[0]
        self.weights = self.weights[1:]

In [37]:
model_elastic = Lasso()
model_elastic.fit(X_train, y_train)
y_pred_train_elastic = model_elastic.predict(X_train)
y_pred_test_elastic = model_elastic.predict(X_test)

In [38]:
model_elareg = ElasticNetRegression()
model_elareg.fit(X_train, y_train)
y_pred_train_elareg = model_elareg.predict(X_train)
y_pred_test_elareg = model_elareg.predict(X_test)

In [39]:
result_MAE.loc[6] = ['ElasticNet default', mae(y_train, y_pred_train_elastic), mae(y_test, y_pred_test_elastic)]
result_RMSE.loc[6] = ['ElasticNet default', rmse(y_train, y_pred_train_elastic), rmse(y_test, y_pred_test_elastic)]
result_R2.loc[6] = ['ElasticNet default', r2_score(y_train, y_pred_train_elastic), r2_score(y_test, y_pred_test_elastic)]

In [40]:
result_MAE.loc[7] = ['ElasticNet', mae(y_train, y_pred_train_elareg), mae(y_test, y_pred_test_elareg)]
result_RMSE.loc[7] = ['ElasticNet', rmse(y_train, y_pred_train_elareg), rmse(y_test, y_pred_test_elareg)]
result_R2.loc[7] = ['ElasticNet', r2_score(y_train, y_pred_train_elareg), r2_score(y_test, y_pred_test_elareg)]

In [41]:
result_MAE

,model,train,test
0,Linreg default,694.523900,662.951989
1,Linreg DSGD,693.439133,661.769381
2,Ridge default,694.523849,662.949312
3,Ridge,692.914087,659.752965
4,Lasso default,694.381215,662.818171
5,Lasso,693.794974,660.949370
6,ElasticNet default,694.381215,662.818171
7,ElasticNet,693.695545,660.815361


In [42]:
result_RMSE

,model,train,test
0,Linreg default,968.869323,890.118262
1,Linreg DSGD,968.902286,889.826433
2,Ridge default,968.869328,890.109990
3,Ridge,970.791588,886.502653
4,Lasso default,968.918533,889.823002
5,Lasso,969.962851,886.893415
6,ElasticNet default,968.918533,889.823002
7,ElasticNet,970.030536,886.834180


In [43]:
result_R2

,model,train,test
0,Linreg default,0.549336,0.512232
1,Linreg DSGD,0.549305,0.512552
2,Ridge default,0.549336,0.512241
3,Ridge,0.547546,0.516187
4,Lasso default,0.549290,0.512556
5,Lasso,0.548318,0.515760
6,ElasticNet default,0.549290,0.512556
7,ElasticNet,0.548255,0.515825


### 6) Feature normalization

#### Several examples of why and where feature normalization is mandatory and vice versa
Нормализация нужна для приведения признаков к одному масштабу, например, один признак измеряется в миллионах (цена квартиры), а другой в десятках (площадь квартиры), т.е. нормализация нужна там, где высчитывается какое то расстояние, например, для линейной регресси, кластеризации ит.д. А где то масштаб наших признаков не важен, например, в деревьях решений

Mathematical formula for MinMaxScaler
$$
X_{std} = \frac{X - X_{min}}{X_{max} - X_{min}}
$$
$$
X_{scaled} = X_{std} * (max - min) + min
$$

In [44]:
class MinMax:
    def __init__(self):
        self.X_min = None
        self.X_max = None
    def fit(self, X):
        X = np.array(X)
        self.X_min = np.min(X, axis=0)
        self.X_max = np.max(X, axis=0)
    def transform(self, X):
        X = np.array(X)
        X_scaled = (X - self.X_min) / (self.X_max - self.X_min + 1e-14)
        return X_scaled

Mathematical formula for StandardScaler
$$
z = \frac{x - u}{s}
$$

In [45]:
class Standard:
    def __init__(self):
        self.mean = None
        self.std = None
    def fit(self, X):
        X = np.array(X)
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
    def transform(self, X):
        X = np.array(X)
        X_scaled = (X - self.mean) / (self.std + 1e-14)
        return X_scaled

### 7) Fit custom and sklearn models with normalized data

In [46]:
scaler = MinMaxScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [47]:
minmaxscaler = MinMax()
minmaxscaler.fit(X_train)
X_train_MinMax = minmaxscaler.transform(X_train)
X_test_MinMax = minmaxscaler.transform(X_test)

In [48]:
model.fit(X_train_scaled, y_train)
y_pred_train_def = model.predict(X_train_scaled)
y_pred_test_def = model.predict(X_test_scaled)

In [49]:
dsgd_model.fit(X_train_scaled, y_train)
y_pred_train_dsgd = dsgd_model.predict(X_train_MinMax)
y_pred_test_dsgd = dsgd_model.predict(X_test_MinMax)

In [50]:
result_MAE.loc[8] = ['Linreg MinMaxScaler', mae(y_train, y_pred_train_def), mae(y_test, y_pred_test_def)]
result_RMSE.loc[8] = ['Linreg MinMaxScaler', rmse(y_train, y_pred_train_def), rmse(y_test, y_pred_test_def)]
result_R2.loc[8] = ['Linreg MinMaxScaler', r2_score(y_train, y_pred_train_def), r2_score(y_test, y_pred_test_def)]

In [51]:
result_MAE.loc[9] = ['Linreg DSGD MinMaxScaler', mae(y_train, y_pred_train_dsgd), mae(y_test, y_pred_test_dsgd)]
result_RMSE.loc[9] = ['Linreg DSGD MinMaxScaler', rmse(y_train, y_pred_train_dsgd), rmse(y_test, y_pred_test_dsgd)]
result_R2.loc[9] = ['Linreg DSGD MinMaxScaler', r2_score(y_train, y_pred_train_dsgd), r2_score(y_test, y_pred_test_dsgd)]

In [52]:
model_ridge.fit(X_train_scaled, y_train)
y_pred_train_ridge = model_ridge.predict(X_train_scaled)
y_pred_test_ridge = model_ridge.predict(X_test_scaled)

In [53]:
model_ridreg.fit(X_train_scaled, y_train)
y_pred_train_ridreg = model_ridreg.predict(X_train_MinMax)
y_pred_test_ridreg = model_ridreg.predict(X_test_MinMax)

In [54]:
result_MAE.loc[10] = ['Ridge MinMaxScaler', mae(y_train, y_pred_train_ridge), mae(y_test, y_pred_test_ridge)]
result_RMSE.loc[10] = ['Ridge MinMaxScaler', rmse(y_train, y_pred_train_ridge), rmse(y_test, y_pred_test_ridge)]
result_R2.loc[10] = ['Ridge MinMaxScaler', r2_score(y_train, y_pred_train_ridge), r2_score(y_test, y_pred_test_ridge)]

In [55]:
result_MAE.loc[11] = ['Ridge MinMax', mae(y_train, y_pred_train_ridreg), mae(y_test, y_pred_test_ridreg)]
result_RMSE.loc[11] = ['Ridge MinMax', rmse(y_train, y_pred_train_ridreg), rmse(y_test, y_pred_test_ridreg)]
result_R2.loc[11] = ['Ridge MinMax', r2_score(y_train, y_pred_train_ridreg), r2_score(y_test, y_pred_test_ridreg)]

In [56]:
model_lasso.fit(X_train_scaled, y_train)
y_pred_train_lasso = model_lasso.predict(X_train_scaled)
y_pred_test_lasso = model_lasso.predict(X_test_scaled)

In [57]:
model_lasreg.fit(X_train_scaled, y_train)
y_pred_train_lasreg = model_lasreg.predict(X_train_MinMax)
y_pred_test_lasreg = model_lasreg.predict(X_test_MinMax)

In [58]:
result_MAE.loc[12] = ['Lasso MinMaxScaler', mae(y_train, y_pred_train_lasso), mae(y_test, y_pred_test_lasso)]
result_RMSE.loc[12] = ['Lasso MinMaxScaler', rmse(y_train, y_pred_train_lasso), rmse(y_test, y_pred_test_lasso)]
result_R2.loc[12] = ['Lasso MinMaxScaler', r2_score(y_train, y_pred_train_lasso), r2_score(y_test, y_pred_test_lasso)]

In [59]:
result_MAE.loc[13] = ['Lasso MinMax', mae(y_train, y_pred_train_lasreg), mae(y_test, y_pred_test_lasreg)]
result_RMSE.loc[13] = ['Lasso MinMax', rmse(y_train, y_pred_train_lasreg), rmse(y_test, y_pred_test_lasreg)]
result_R2.loc[13] = ['Lasso MinMax', r2_score(y_train, y_pred_train_lasreg), r2_score(y_test, y_pred_test_lasreg)]

In [60]:
model_elastic.fit(X_train_scaled, y_train)
y_pred_train_elastic = model_elastic.predict(X_train_scaled)
y_pred_test_elastic = model_elastic.predict(X_test_scaled)

In [61]:
model_elareg.fit(X_train_scaled, y_train)
y_pred_train_elareg = model_elareg.predict(X_train_MinMax)
y_pred_test_elareg = model_elareg.predict(X_test_MinMax)

In [62]:
result_MAE.loc[14] = ['ElasticNet MinMaxScaler', mae(y_train, y_pred_train_elastic), mae(y_test, y_pred_test_elastic)]
result_RMSE.loc[14] = ['ElasticNet MinMaxScaler', rmse(y_train, y_pred_train_elastic), rmse(y_test, y_pred_test_elastic)]
result_R2.loc[14] = ['ElasticNet MinMaxScaler', r2_score(y_train, y_pred_train_elastic), r2_score(y_test, y_pred_test_elastic)]

In [63]:
result_MAE.loc[15] = ['ElasticNet MinMax', mae(y_train, y_pred_train_elareg), mae(y_test, y_pred_test_elareg)]
result_RMSE.loc[15] = ['ElasticNet MinMax', rmse(y_train, y_pred_train_elareg), rmse(y_test, y_pred_test_elareg)]
result_R2.loc[15] = ['ElasticNet MinMax', r2_score(y_train, y_pred_train_elareg), r2_score(y_test, y_pred_test_elareg)]

In [64]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [65]:
standard = Standard()
standard.fit(X_train)
X_train_standard = standard.transform(X_train)
X_test_standard = standard.transform(X_test)

In [66]:
model.fit(X_train_scaled, y_train)
y_pred_train_def = model.predict(X_train_scaled)
y_pred_test_def = model.predict(X_test_scaled)

In [67]:
dsgd_model.fit(X_train_scaled, y_train)
y_pred_train_dsgd = dsgd_model.predict(X_train_standard)
y_pred_test_dsgd = dsgd_model.predict(X_test_standard)

In [68]:
result_MAE.loc[16] = ['Linreg StandardScaler', mae(y_train, y_pred_train_def), mae(y_test, y_pred_test_def)]
result_RMSE.loc[16] = ['Linreg StandardScaler', rmse(y_train, y_pred_train_def), rmse(y_test, y_pred_test_def)]
result_R2.loc[16] = ['Linreg StandardScaler', r2_score(y_train, y_pred_train_def), r2_score(y_test, y_pred_test_def)]

In [69]:
result_MAE.loc[17] = ['Linreg DSGD StandardScaler', mae(y_train, y_pred_train_dsgd), mae(y_test, y_pred_test_dsgd)]
result_RMSE.loc[17] = ['Linreg DSGD StandardScaler', rmse(y_train, y_pred_train_dsgd), rmse(y_test, y_pred_test_dsgd)]
result_R2.loc[17] = ['Linreg DSGD StandardScaler', r2_score(y_train, y_pred_train_dsgd), r2_score(y_test, y_pred_test_dsgd)]

In [70]:
model_ridge.fit(X_train_scaled, y_train)
y_pred_train_ridge = model_ridge.predict(X_train_scaled)
y_pred_test_ridge = model_ridge.predict(X_test_scaled)

In [71]:
model_ridreg.fit(X_train_scaled, y_train)
y_pred_train_ridreg = model_ridreg.predict(X_train_standard)
y_pred_test_ridreg = model_ridreg.predict(X_test_standard)

In [72]:
result_MAE.loc[18] = ['Ridge StandardScaler', mae(y_train, y_pred_train_ridge), mae(y_test, y_pred_test_ridge)]
result_RMSE.loc[18] = ['Ridge StandardScaler', rmse(y_train, y_pred_train_ridge), rmse(y_test, y_pred_test_ridge)]
result_R2.loc[18] = ['Ridge StandardScaler', r2_score(y_train, y_pred_train_ridge), r2_score(y_test, y_pred_test_ridge)]

In [73]:
result_MAE.loc[19] = ['Ridge Standard', mae(y_train, y_pred_train_ridreg), mae(y_test, y_pred_test_ridreg)]
result_RMSE.loc[19] = ['Ridge Standard', rmse(y_train, y_pred_train_ridreg), rmse(y_test, y_pred_test_ridreg)]
result_R2.loc[19] = ['Ridge Standard', r2_score(y_train, y_pred_train_ridreg), r2_score(y_test, y_pred_test_ridreg)]

In [74]:
model_lasso.fit(X_train_scaled, y_train)
y_pred_train_lasso = model_lasso.predict(X_train_scaled)
y_pred_test_lasso = model_lasso.predict(X_test_scaled)

In [75]:
model_lasreg.fit(X_train_scaled, y_train)
y_pred_train_lasreg = model_lasreg.predict(X_train_standard)
y_pred_test_lasreg = model_lasreg.predict(X_test_standard)

In [76]:
result_MAE.loc[20] = ['Lasso StandardScaler', mae(y_train, y_pred_train_lasso), mae(y_test, y_pred_test_lasso)]
result_RMSE.loc[20] = ['Lasso StandardScaler', rmse(y_train, y_pred_train_lasso), rmse(y_test, y_pred_test_lasso)]
result_R2.loc[20] = ['Lasso StandardScaler', r2_score(y_train, y_pred_train_lasso), r2_score(y_test, y_pred_test_lasso)]

In [77]:
result_MAE.loc[21] = ['Lasso Standard', mae(y_train, y_pred_train_lasreg), mae(y_test, y_pred_test_lasreg)]
result_RMSE.loc[21] = ['Lasso Standard', rmse(y_train, y_pred_train_lasreg), rmse(y_test, y_pred_test_lasreg)]
result_R2.loc[21] = ['Lasso Standard', r2_score(y_train, y_pred_train_lasreg), r2_score(y_test, y_pred_test_lasreg)]

In [78]:
model_elastic.fit(X_train_scaled, y_train)
y_pred_train_elastic = model_elastic.predict(X_train_scaled)
y_pred_test_elastic = model_elastic.predict(X_test_scaled)

In [79]:
model_elareg.fit(X_train_scaled, y_train)
y_pred_train_elareg = model_elareg.predict(X_train_standard)
y_pred_test_elareg = model_elareg.predict(X_test_standard)

In [80]:
result_MAE.loc[22] = ['ElasticNet StandardScaler', mae(y_train, y_pred_train_elastic), mae(y_test, y_pred_test_elastic)]
result_RMSE.loc[22] = ['ElasticNet StandardScaler', rmse(y_train, y_pred_train_elastic), rmse(y_test, y_pred_test_elastic)]
result_R2.loc[22] = ['ElasticNet StandardScaler', r2_score(y_train, y_pred_train_elastic), r2_score(y_test, y_pred_test_elastic)]

In [81]:
result_MAE.loc[23] = ['ElasticNet Standard', mae(y_train, y_pred_train_elareg), mae(y_test, y_pred_test_elareg)]
result_RMSE.loc[23] = ['ElasticNet Standard', rmse(y_train, y_pred_train_elareg), rmse(y_test, y_pred_test_elareg)]
result_R2.loc[23] = ['ElasticNet Standard', r2_score(y_train, y_pred_train_elareg), r2_score(y_test, y_pred_test_elareg)]

In [82]:
result_MAE

,model,train,test
0,Linreg default,694.523900,662.951989
1,Linreg DSGD,693.439133,661.769381
2,Ridge default,694.523849,662.949312
3,Ridge,692.914087,659.752965
4,Lasso default,694.381215,662.818171
5,Lasso,693.794974,660.949370
6,ElasticNet default,694.381215,662.818171
7,ElasticNet,693.695545,660.815361
8,Linreg MinMaxScaler,694.523900,662.951989
9,Linreg DSGD MinMaxScaler,692.982182,661.302554


In [83]:
result_RMSE

,model,train,test
0,Linreg default,968.869323,890.118262
1,Linreg DSGD,968.902286,889.826433
2,Ridge default,968.869328,890.109990
3,Ridge,970.791588,886.502653
4,Lasso default,968.918533,889.823002
5,Lasso,969.962851,886.893415
6,ElasticNet default,968.918533,889.823002
7,ElasticNet,970.030536,886.834180
8,Linreg MinMaxScaler,968.869323,890.118262
9,Linreg DSGD MinMaxScaler,968.930435,889.775774


In [84]:
result_R2

,model,train,test
0,Linreg default,0.549336,0.512232
1,Linreg DSGD,0.549305,0.512552
2,Ridge default,0.549336,0.512241
3,Ridge,0.547546,0.516187
4,Lasso default,0.549290,0.512556
5,Lasso,0.548318,0.515760
6,ElasticNet default,0.549290,0.512556
7,ElasticNet,0.548255,0.515825
8,Linreg MinMaxScaler,0.549336,0.512232
9,Linreg DSGD MinMaxScaler,0.549279,0.512607


### 8) Overfit models

In [85]:
X_train = data_train[['bathrooms', 'bedrooms']]
y_train = data_train['price']
X_test = data_test[['bathrooms', 'bedrooms']]
y_test = data_test['price']

In [86]:
poly = PolynomialFeatures(degree=10)
poly.fit(X_train)
X_train_poly = poly.transform(X_train)
X_test_poly = poly.transform(X_test)

In [87]:
model.fit(X_train_poly, y_train)
y_pred_train_def = model.predict(X_train_poly)
y_pred_test_def = model.predict(X_test_poly)

In [88]:
result_MAE.loc[24] = ['Linreg Polynomial', mae(y_train, y_pred_train_def), mae(y_test, y_pred_test_def)]
result_RMSE.loc[24] = ['Linreg Polynomial', rmse(y_train, y_pred_train_def), rmse(y_test, y_pred_test_def)]
result_R2.loc[24] = ['Linreg Polynomial', r2_score(y_train, y_pred_train_def), r2_score(y_test, y_pred_test_def)]

In [89]:
model_ridge.fit(X_train_poly, y_train)
y_pred_train_ridge = model_ridge.predict(X_train_poly)
y_pred_test_ridge = model_ridge.predict(X_test_poly)

/home/dRAGON/Desktop/Projects21/ML.Project_2.ID_1254799-1/src/jupyter/lib/python3.13/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=7.72395e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


In [90]:
result_MAE.loc[25] = ['Ridge Polynomial', mae(y_train, y_pred_train_ridge), mae(y_test, y_pred_test_ridge)]
result_RMSE.loc[25] = ['Ridge Polynomial', rmse(y_train, y_pred_train_ridge), rmse(y_test, y_pred_test_ridge)]
result_R2.loc[25] = ['Ridge Polynomial', r2_score(y_train, y_pred_train_ridge), r2_score(y_test, y_pred_test_ridge)]

In [91]:
model_lasso.fit(X_train_poly, y_train)
y_pred_train_lasso = model_lasso.predict(X_train_poly)
y_pred_test_lasso = model_lasso.predict(X_test_poly)

/home/dRAGON/Desktop/Projects21/ML.Project_2.ID_1254799-1/src/jupyter/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.452e+10, tolerance: 9.906e+06
  model = cd_fast.enet_coordinate_descent(


In [92]:
result_MAE.loc[26] = ['Lasso Polynomial', mae(y_train, y_pred_train_lasso), mae(y_test, y_pred_test_lasso)]
result_RMSE.loc[26] = ['Lasso Polynomial', rmse(y_train, y_pred_train_lasso), rmse(y_test, y_pred_test_lasso)]
result_R2.loc[26] = ['Lasso Polynomial', r2_score(y_train, y_pred_train_lasso), r2_score(y_test, y_pred_test_lasso)]

In [ ]:
model_elastic.fit(X_train_poly, y_train)
y_pred_train_elastic = model_elastic.predict(X_train_poly)
y_pred_test_elastic = model_elastic.predict(X_test_poly)

In [ ]:
result_MAE.loc[27] = ['ElasticNet Polynomial', mae(y_train, y_pred_train_elastic), mae(y_test, y_pred_test_elastic)]
result_RMSE.loc[27] = ['ElasticNet Polynomial', rmse(y_train, y_pred_train_elastic), rmse(y_test, y_pred_test_elastic)]
result_R2.loc[27] = ['ElasticNet Polynomial', r2_score(y_train, y_pred_train_elastic), r2_score(y_test, y_pred_test_elastic)]

In [ ]:
result_MAE

In [ ]:
result_RMSE

In [ ]:
result_R2

### 9) Native models

In [ ]:
y_mean = y_train.mean()
y_median = y_train.median()

In [ ]:
y_pred_mean_train = np.full_like(y_train, y_mean)
y_pred_mean_test = np.full_like(y_test, y_mean)

In [ ]:
y_pred_median_train = np.full_like(y_train, y_median)
y_pred_median_test = np.full_like(y_test, y_median)

In [ ]:
result_MAE.loc[len(result_MAE)] = ['Native Mean', mae(y_train, y_pred_mean_train), mae(y_test, y_pred_mean_test)]
result_RMSE.loc[len(result_RMSE)] = ['Native Mean', rmse(y_train, y_pred_mean_train), rmse(y_test, y_pred_mean_test)]
result_R2.loc[len(result_R2)] = ['Native Mean', r2_score(y_train, y_pred_mean_train), r2_score(y_test, y_pred_mean_test)]

In [ ]:
result_MAE.loc[len(result_MAE)] = ['Native Median', mae(y_train, y_pred_median_train), mae(y_test, y_pred_median_test)]
result_RMSE.loc[len(result_RMSE)] = ['Native Median', rmse(y_train, y_pred_median_train), rmse(y_test, y_pred_median_test)]
result_R2.loc[len(result_R2)] = ['Native Median', r2_score(y_train, y_pred_median_train), r2_score(y_test, y_pred_median_test)]

### 10) Compare results

In [ ]:
result_MAE = result_MAE.round(6)
result_RMSE = result_RMSE.round(6)
result_R2 = result_R2.round(6)

In [ ]:
result_MAE

In [ ]:
result_RMSE

In [ ]:
result_R2

Лучшая модель - это Linreg default \
Самая стабильная модель - это Ridge default

### 11) Addition task

In [ ]:
y_train_log = np.log1p(y_train)

In [ ]:
model_log = Ridge()
model_log.fit(X_train_scaled, y_train_log)
y_pred_log = model_log.predict(X_test_scaled)
y_pred_final = np.expm1(y_pred_log)

In [ ]:
print(f"MAE: {mae(y_test, y_pred_final)}")
print(f"RMSE: {rmse(y_test, y_pred_final)}")
print(f"R2: {r2_score(y_test, y_pred_final)}")